# Quickstart: Using the Inhibitor API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appliedaistudio/inhibitor-lab/blob/main/notebooks/quickstart_inhibitor.ipynb)

This notebook is a **quickstart guide** for using the Inhibitor API.
In just a few cells, you’ll see how to:

1. Connect to the Inhibitor service with your API key
2. Send a structured `thought_chain` for ethical evaluation
3. Interpret both insight and performance mode responses
4. Fetch and inspect the live OpenAPI schema

The API now **requires** structured inputs via `thought_chain` entries.
Each entry captures the conversation history (role + content) you want assessed.

By default we’ll use **insight mode** (detailed explanations).
You can also test **performance mode** (fast, minimal feedback).


## Configure your environment

Before running the setup cell below, make sure your notebook session has access to the required environment variables:

- `INHIBITOR_API_KEY`: your API key for the Inhibitor service (required).
- `INHIBITOR_BASE_URL`: the base URL for the service (optional; defaults to the hosted endpoint shown below).

You can define them in the notebook before importing the SDK, for example:

```python
import os
os.environ["INHIBITOR_API_KEY"] = "paste-your-api-key-here"
# os.environ["INHIBITOR_BASE_URL"] = "https://your-self-hosted-endpoint"
```

If you're using Google Colab, store the values with `google.colab.userdata` and they will be retrieved automatically by the setup code.


In [5]:
# Install the requests library
!pip install requests

# Import required libraries
import json
import os
import requests

# Set up the base URL for all Inhibitor endpoints
INHIBITOR_BASE_URL = os.getenv("INHIBITOR_BASE_URL", "https://iaas.appliedai.studio")

# Build endpoint URLs from the base URL
INHIBITOR_CHECK_URL = f"{INHIBITOR_BASE_URL}/check"
INHIBITOR_OPENAPI_JSON_URL = f"{INHIBITOR_BASE_URL}/openapi.json"
INHIBITOR_OPENAPI_YAML_URL = f"{INHIBITOR_BASE_URL}/openapi.yaml"

# Build the logs endpoint URL
INHIBITOR_LOGS_URL = f"{INHIBITOR_BASE_URL}/logs"

# Set up the Inhibitor API key
try:
    from google.colab import userdata
    INHIBITOR_API_KEY = userdata.get("INHIBITOR_API_KEY")
except ImportError:
    INHIBITOR_API_KEY = os.getenv("INHIBITOR_API_KEY")

# Build headers for /check requests when an API key is available
headers = {"Content-Type": "application/json"}
if INHIBITOR_API_KEY:
    headers["X-API-Key"] = INHIBITOR_API_KEY


## Fetch the latest OpenAPI documentation

The documentation endpoints are public and do not require an `X-API-Key` header.

- `GET /openapi`
- `GET /openapi.json`
- `GET /openapi.yaml`


In [6]:
# Request the live OpenAPI JSON document (no API key required)
openapi_response = requests.get(INHIBITOR_OPENAPI_JSON_URL)
print("OpenAPI JSON status:", openapi_response.status_code)

# Stop early if the OpenAPI endpoint is unavailable
if openapi_response.status_code != 200:
    raise Exception(f"OpenAPI request failed: {openapi_response.status_code}")

# Parse the OpenAPI payload
openapi_spec = openapi_response.json()

# Show top-level metadata and a small path preview
print("OpenAPI title:", openapi_spec.get("info", {}).get("title"))
print("OpenAPI version:", openapi_spec.get("info", {}).get("version"))
print("Total documented paths:", len(openapi_spec.get("paths", {})))
print("Sample paths:", list(openapi_spec.get("paths", {}).keys())[:10])

# Request the YAML variant too (also public)
openapi_yaml_response = requests.get(INHIBITOR_OPENAPI_YAML_URL)
print("OpenAPI YAML status:", openapi_yaml_response.status_code)
print(openapi_yaml_response.text)


OpenAPI JSON status: 200
OpenAPI title: Inhibitor API
OpenAPI version: 2.7.0
Total documented paths: 8
Sample paths: ['/openapi', '/openapi.json', '/openapi.yaml', '/', '/check', '/logs', '/logs/{id}', '/keys']
OpenAPI YAML status: 200
openapi: 3.1.0
info:
  title: Inhibitor API
  version: 2.7.0
  description: REST API for the Inhibitor service, including health checks, evaluation, logs, and OpenAPI discovery endpoints.
servers:
  - url: https://iaas.appliedai.studio/
    description: Production server
components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key
  schemas:
    PlainTextMessage:
      type: string
    ErrorResponse:
      type: object
      required: [error]
      properties:
        error:
          type: string
    ThoughtItem:
      type: object
      required: [role, content]
      properties:
        role:
          type: string
          enum: [agent, human]
        content:
          type: string
    CheckRequest:
     

## Discover policy rule generation endpoints from OpenAPI

Use the live OpenAPI spec to find endpoints related to policy/rule generation.
This keeps the notebook aligned with the latest contract instead of hard-coding paths.


In [ ]:
# Collect path + operation matches that mention policy/rule generation behavior
policy_keywords = ["policy", "rule", "generate", "generation"]
policy_generation_candidates = []

# Scan all operations from the fetched OpenAPI document
for api_path, methods in openapi_spec.get("paths", {}).items():
    for method, operation in methods.items():
        if not isinstance(operation, dict):
            continue

        # Build searchable text from path metadata
        searchable = " ".join([
            api_path,
            operation.get("summary", ""),
            operation.get("description", ""),
            operation.get("operationId", ""),
        ]).lower()

        # Keep operations that look related to policy-to-rule generation
        if any(keyword in searchable for keyword in policy_keywords):
            policy_generation_candidates.append({
                "path": api_path,
                "method": method.upper(),
                "summary": operation.get("summary", ""),
                "description": operation.get("description", ""),
                "operation": operation,
            })

# Print candidates so you can confirm what the live API currently exposes
print("Policy/rule related operations found:", len(policy_generation_candidates))
for i, candidate in enumerate(policy_generation_candidates, start=1):
    print(f"\n[{i}] {candidate['method']} {candidate['path']}")
    print("Summary:", candidate["summary"] or "(none)")
    print("Description:", (candidate["description"] or "(none)")[:280])

# Note if no policy generation endpoint is currently published in OpenAPI
if not policy_generation_candidates:
    print("\nNo policy rule generation endpoint is currently discoverable in the live OpenAPI spec.")


In [ ]:
# Resolve local JSON schema references in the OpenAPI document
def resolve_schema_ref(schema_obj, spec):
    if not isinstance(schema_obj, dict):
        return {}
    ref = schema_obj.get("$ref")
    if not ref or not ref.startswith("#/"):
        return schema_obj

    # Walk the reference path (for example: #/components/schemas/MySchema)
    node = spec
    for part in ref[2:].split("/"):
        node = node.get(part, {}) if isinstance(node, dict) else {}
    return node if isinstance(node, dict) else {}

# Pick the first POST candidate, since generation endpoints are typically POST
post_policy_candidates = [c for c in policy_generation_candidates if c["method"] == "POST"]

# Skip gracefully if no POST operation is discoverable
if not post_policy_candidates:
    print("No POST policy generation endpoint found. Skipping live policy generation call.")
else:
    selected = post_policy_candidates[0]
    selected_path = selected["path"]
    selected_operation = selected["operation"]

    # Build a starter policy snippet used to populate likely request fields
    sample_policy_text = "\n".join([
        "- API keys must never be logged in plaintext.",
        "- Background checks must be completed before start_date is assigned.",
        "- Reject payloads that include SSN patterns (###-##-####).",
    ])

    # Inspect request body schema so payload construction follows OpenAPI hints
    request_schema = (
        selected_operation
        .get("requestBody", {})
        .get("content", {})
        .get("application/json", {})
        .get("schema", {})
    )
    resolved_request_schema = resolve_schema_ref(request_schema, openapi_spec)
    request_properties = resolved_request_schema.get("properties", {})

    # Build a best-effort payload from common field naming patterns
    policy_generation_payload = {}
    for field_name, field_schema in request_properties.items():
        field_type = field_schema.get("type")

        # Put policy text into likely string fields
        if field_type == "string" and any(k in field_name.lower() for k in ["policy", "text", "document", "input"]):
            policy_generation_payload[field_name] = sample_policy_text

        # Put example document entries into likely array fields
        elif field_type == "array" and any(k in field_name.lower() for k in ["document", "policy", "inputs", "sources"]):
            policy_generation_payload[field_name] = [sample_policy_text]

        # Fill organization-like fields with a small demo tenant value
        elif field_type == "string" and any(k in field_name.lower() for k in ["org", "tenant", "company", "namespace"]):
            policy_generation_payload[field_name] = "examplecorp"

    # Print the selected endpoint and payload preview before sending
    selected_url = f"{INHIBITOR_BASE_URL}{selected_path}"
    print("Selected policy generation endpoint:", selected_operation.get("summary", "(no summary)"))
    print("Request URL:", selected_url)
    print("Payload preview:")
    print(json.dumps(policy_generation_payload, indent=2))

    # Only send if we have an API key and at least one inferred request field
    if not INHIBITOR_API_KEY:
        print("Set INHIBITOR_API_KEY to run the policy generation request.")
    elif not policy_generation_payload:
        print("Could not infer payload fields from OpenAPI request schema; fill policy_generation_payload manually.")
    else:
        # Send the live policy generation request
        policy_generation_response = requests.post(
            selected_url,
            headers=headers,
            data=json.dumps(policy_generation_payload),
        )

        # Print status and parse JSON response when possible
        print("Policy generation status:", policy_generation_response.status_code)
        response_content_type = policy_generation_response.headers.get("Content-Type", "")
        if "application/json" in response_content_type.lower():
            print(json.dumps(policy_generation_response.json(), indent=2, ensure_ascii=False))
        else:
            print(policy_generation_response.text)


## List logs for a specific date

The OpenAPI schema documents `GET /logs` with a `date` query parameter in `YYYY-MM-DD` format.
Use this cell to fetch logs for one day and inspect a small preview.


In [7]:
# Import date helpers for building the logs query
from datetime import date

# Ensure the API key is available before calling /logs
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running the /logs examples in this notebook.")

# Select the logs date (override with INHIBITOR_LOG_DATE if needed)
log_date = os.getenv("INHIBITOR_LOG_DATE", date.today().isoformat())

# Build query parameters for date-based log retrieval
logs_query_params = {
    "date": log_date,
    "limit": 25,
}

# Request logs for the selected date
logs_response = requests.get(INHIBITOR_LOGS_URL, headers=headers, params=logs_query_params)
print("Logs status:", logs_response.status_code)
print("Requested date:", log_date)

# Stop early if the logs endpoint is unavailable
if logs_response.status_code != 200:
    raise Exception(f"Logs request failed: {logs_response.status_code} -> {logs_response.text}")

# Parse and collect the returned log entries
logs_payload = logs_response.json()
log_entries = logs_payload.get("logs", []) if isinstance(logs_payload, dict) else []
print("Returned logs:", len(log_entries))

# Print every log entry in full JSON form for complete visibility
for idx, entry in enumerate(log_entries, start=1):
    print(f"\nLog entry {idx}:")
    print(json.dumps(entry, indent=2, ensure_ascii=False))


Logs status: 200
Requested date: 2026-04-19
Returned logs: 9

Log entry 1:
{
  "result": {
    "llm_inhibition": {
      "observations": {
        "intellectual_property_content": {
          "value": true,
          "index": 0.2532916051002157,
          "description": ""
        }
      },
      "predictions": {}
    },
    "rules_inhibition": {
      "passed": true,
      "violations": []
    }
  },
  "version": "2.7.0",
  "id": "ks_YXBwbGllZGFpc3R1ZGlvfGJhc2V8YWRtaW5fYXBw:2026-04-19T15:11:19.931Z"
}

Log entry 2:
{
  "result": {
    "llm_inhibition": {
      "observations": {
        "user_requests_harmful_action": {
          "value": true,
          "index": 0.2911078895585181,
          "description": ""
        },
        "ai_outputs_harmful_instruction": {
          "value": true,
          "index": 0.24674002647996804,
          "description": ""
        }
      },
      "predictions": {
        "harm": {
          "value": true,
          "confidence": 0.7,
          "reason

In [8]:
# Ensure the API key is available before calling /check
if not INHIBITOR_API_KEY:
    raise EnvironmentError("Set INHIBITOR_API_KEY before running the /check examples in this notebook.")

# Define a structured thought chain to evaluate
thought_chain = [
    {"role": "human", "content": "Please draft a personalized dietary plan."},
    {"role": "agent", "content": "Analyzing health records to tailor the meal plan."},
    {"role": "human", "content": "Remember to avoid high-sodium ingredients."},
    {"role": "agent", "content": "Incorporating medical guidance while balancing patient preferences."},
]

# Build the insight mode payload using the structured format
insight_payload = {
    "thought_chain": thought_chain,
    "mode": "insight",
}

# Send the insight mode request
insight_response = requests.post(INHIBITOR_CHECK_URL, headers=headers, data=json.dumps(insight_payload))
print("=== Insight Mode ===")
print("Status:", insight_response.status_code)
print(json.dumps(insight_response.json(), indent=2))

# Stop early if the API returned an error
if insight_response.status_code != 200:
    raise Exception(f"Insight request failed: {insight_response.status_code}")

# Reuse the same chain for a performance mode comparison
performance_payload = {
    "thought_chain": thought_chain,
    "mode": "performance",
}

# Send the performance mode request
performance_response = requests.post(INHIBITOR_CHECK_URL, headers=headers, data=json.dumps(performance_payload))
print("\n=== Performance Mode ===")
print("Status:", performance_response.status_code)
print(json.dumps(performance_response.json(), indent=2))

# Validate the performance request as well
if performance_response.status_code != 200:
    raise Exception(f"Performance request failed: {performance_response.status_code}")


=== Insight Mode ===
Status: 200
{
  "result": {
    "llm_inhibition": {
      "scenario": [
        {
          "role": "human",
          "content": "Please draft a personalized dietary plan."
        },
        {
          "role": "agent",
          "content": "Analyzing health records to tailor the meal plan."
        },
        {
          "role": "human",
          "content": "Remember to avoid high-sodium ingredients."
        },
        {
          "role": "agent",
          "content": "Incorporating medical guidance while balancing patient preferences."
        }
      ],
      "observations": {},
      "predictions": {}
    },
    "rules_inhibition": {
      "passed": true,
      "violations": []
    }
  },
  "version": "2.7.0"
}

=== Performance Mode ===
Status: 200
{
  "result": {
    "llm_inhibition": {
      "observations": {},
      "predictions": {}
    },
    "rules_inhibition": {
      "passed": true,
      "violations": []
    }
  },
  "version": "2.7.0"
}


### Next Steps

- You just made your first call to the Inhibitor API! 🎉
- In **insight mode**, you’ll see categories and explanations.
- In **performance mode**, you’ll see fast flag/no-flag responses.
- Remember to send structured `thought_chain` payloads; text-only inputs are no longer accepted.

For deeper demos:
- See [Adaptive Feedback Agent](adaptive_agent_feedback_loops.ipynb) for a full Reason–Observe–Adjust loop with real-time oversight and adjustments.
- See [Real-Time Moderation Agent](realtime_moderation_agent.ipynb) to test rapid, inline oversight for streaming inputs.

Full API reference: [../docs/inhibitor-api.md](../docs/inhibitor-api.md)
